In [12]:
import sys
import os
sys.path.append(os.path.abspath('../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.cox import *

from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sksurv.metrics import integrated_brier_score
from sksurv.metrics import concordance_index_ipcw
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.inspection import permutation_importance
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis, CoxPHSurvivalAnalysis

import time
set_config(display="text")  # displays text representation of estimators

In [13]:
dataset = pd.read_csv("../data/final/data_final.csv")

diagnosis_variables = []

with open("../data/final/diagnosis_final.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

treatment_variables = []
with open("../data/final/treatment_final.txt", "r") as file:
    for line in file:
        treatment_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = treatment_variables + diagnosis_variables

In [14]:
# Find duplicated patients with regional ln recurrence = 1
mask = (survival_dataset.duplicated(subset='PERSON_ID', keep=False)) #& (survival_dataset['Regional LN recurrence'] == 1)

# For each duplicated PERSON_ID with recurrence, keep only the row with the lowest survival days
to_remove = (
    survival_dataset[mask]
    .sort_values(['PERSON_ID', 'Survival days'], ascending=[True, True])
    .groupby('PERSON_ID')
    .head(-1)
    .index
)

survival_dataset = survival_dataset.drop(to_remove)

In [15]:
# Prepare data
data_x = survival_dataset[feature_columns]

data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)
# Since every duplicated patient has a consistent event indicator (regional ln recurrence) I can use this method
groups = survival_dataset['PERSON_ID']

In [16]:
sgkf = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)
cv = sgkf.split(X=survival_dataset, y=survival_dataset['Regional LN recurrence'])

train_index, test_index = next(cv)

In [17]:
subsampling = survival_dataset.iloc[train_index].copy()

train = survival_dataset.iloc[train_index]
test = survival_dataset.iloc[test_index]

In [ ]:
l1_ratio = 1.0
alpha = 0.0062102
metrics = ['c-index', 'c-index-ipcw', 'int-brier-score']

In [21]:
# Prepare full training and test sets (no subsampling)
X_train = train[feature_columns]
y_train = Surv.from_arrays(
    event=train['Regional LN recurrence'].values,
    time=train['Survival days'].values
)

X_test = test[feature_columns]
y_test = Surv.from_arrays(
    event=test['Regional LN recurrence'].values,
    time=test['Survival days'].values
)

print(f"Training set size: {len(X_train)} patients")
print(f"  Events: {y_train['event'].sum()} out of {len(y_train)} ({y_train['event'].mean()*100:.1f}%)")
print(f"Test set size: {len(X_test)} patients")
print(f"  Events: {y_test['event'].sum()} out of {len(y_test)} ({y_test['event'].mean()*100:.1f}%)")

# Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the Cox model with chosen hyperparameters
print(f"\nTraining Cox model with:")
print(f"  alpha = {alpha}")
print(f"  l1_ratio = {l1_ratio}")

model = CoxnetSurvivalAnalysis(
    alphas=[alpha], 
    l1_ratio=l1_ratio, 
    fit_baseline_model=True, 
    max_iter=200000
)
model.fit(X_train_scaled, y_train)

print("\nModel training complete!")
print(f"Number of non-zero coefficients: {np.sum(model.coef_[0] != 0)}")

Training set size: 2376 patients
  Events: 27 out of 2376 (1.1%)
Test set size: 594 patients
  Events: 6 out of 594 (1.0%)

Training Cox model with:
  alpha = 0.0062102
  l1_ratio = 1.0

Model training complete!
Number of non-zero coefficients: 0


In [ ]:
from lifelines import CoxPHFitter
import numpy as np

# Calculate linear predictor using fixed coefficients
feature_names = list(fixed_coefficients.keys())
coefficients = np.array(list(fixed_coefficients.values()))

# Calculate the linear predictor (risk score) using fixed coefficients
linear_predictor = train[feature_names].dot(coefficients)

print("Fixed coefficients being used:")
for feature, coef in fixed_coefficients.items():
    print(f"  {feature}: {coef:.4f}")

print(f"\nLinear predictor calculated for {len(train)} samples")
print(f"Linear predictor range: [{linear_predictor.min():.4f}, {linear_predictor.max():.4f}]")

# Create data for baseline hazard estimation using the BreslowEstimator approach
# We'll use the linear predictor as the risk score

# Extract survival times and events
survival_times = train['Survival days'].values
events = train['Regional LN recurrence'].values

print(f"\nSurvival data summary:")
print(f"  Time range: {survival_times.min()} to {survival_times.max()} days")
print(f"  Events: {events.sum()} out of {len(events)} ({events.mean()*100:.1f}%)")

# Use Breslow estimator to get baseline hazard with fixed linear predictor
from lifelines.utils import concordance_index
from lifelines import KaplanMeierFitter

# Calculate risk scores (exp of linear predictor)
risk_scores = np.exp(linear_predictor)

# Estimate baseline hazard using the Breslow method
# This is a manual implementation since we want to fix the coefficients
unique_times = np.sort(np.unique(survival_times[events == 1]))
baseline_hazard = []

for t in unique_times:
    # Number of events at time t
    d_t = np.sum((survival_times == t) & (events == 1))
    
    # Risk set at time t (all subjects with survival time >= t)
    at_risk = survival_times >= t
    
    # Sum of risk scores for subjects at risk at time t
    risk_sum = np.sum(risk_scores[at_risk])
    
    # Baseline hazard increment
    if risk_sum > 0:
        h_0_t = d_t / risk_sum
    else:
        h_0_t = 0
    
    baseline_hazard.append(h_0_t)

# Create baseline hazard DataFrame
baseline_hazard_df = pd.DataFrame({
    'timeline': unique_times,
    'baseline_hazard': baseline_hazard
})

# Calculate cumulative baseline hazard
baseline_hazard_df['baseline_cumulative_hazard'] = np.cumsum(baseline_hazard_df['baseline_hazard'])

# Calculate baseline survival function
baseline_hazard_df['baseline_survival'] = np.exp(-baseline_hazard_df['baseline_cumulative_hazard'])

print("\nBaseline Hazard Estimation completed using Breslow method with fixed coefficients")
print("\nBaseline Cumulative Hazard (first 10 time points):")
print(baseline_hazard_df[['timeline', 'baseline_cumulative_hazard']].head(10))

print("\nBaseline Survival Function (first 10 time points):")
print(baseline_hazard_df[['timeline', 'baseline_survival']].head(10))

Fixed coefficients being used:
  Oestrogen receptor status at CB_1.0: 0.9136
  Total administered dose: -0.0285
  Age: 0.0116
  Ki67 positive cells (%) at CB: 0.0115

Linear predictor calculated for 2376 samples
Linear predictor range: [-0.9267, 2.9462]

Survival data summary:
  Time range: 25.0 to 7492.0 days
  Events: 27.0 out of 2376 (1.1%)

Baseline Hazard Estimation completed using Breslow method with fixed coefficients

Baseline Cumulative Hazard (first 10 time points):
   timeline  baseline_cumulative_hazard
0      75.0                    0.000262
1     192.0                    0.000527
2     278.0                    0.000797
3     345.0                    0.001075
4     381.0                    0.001357
5     395.0                    0.001642
6     622.0                    0.001986
7     714.0                    0.002353
8     790.0                    0.002744
9     799.0                    0.003139

Baseline Survival Function (first 10 time points):
   timeline  baseline_survi